In [ ]:
import requests
import json
import pandas as pd
import numpy as np
from pandas import json_normalize
import plotly.graph_objects as go
import plotly.express as px
import yfinance as yf


In [4]:
df = pd.read_csv("Stock_prices_from2012.06 (1).csv", header=[0, 1], index_col=0, parse_dates=True)
df

Price            Close                                                  \
Ticker            AAPL        AMZN       GOOGL        META        MSFT   
Date                                                                     
2012-06-01   16.779005   10.411000   14.154103   27.478691   22.432699   
2012-06-04   16.877703   10.728500   14.342749   26.665831   22.511545   
2012-06-05   16.834030   10.660500   14.139973   25.644798   22.480003   
2012-06-06   17.092157   10.882000   14.391830   26.576614   23.142332   
2012-06-07   17.099934   10.940000   14.333823   26.080971   23.047722   
...                ...         ...         ...         ...         ...   
2026-09-10  326.570007  251.889999  332.600006  644.380005  492.440002   
2026-09-11  332.269989  256.779999  338.500000  648.030029  495.630005   
2026-09-14  333.079987  253.539993  349.390015  665.599976  505.410004   
2026-09-15  331.339996  248.419998  344.980011  670.239990  497.119995   
2026-09-16  332.410004  245.960007  342.869995  673.309998  490.299988   

Price                                              Volume              \
Ticker            NVDA         SPY        TSLA       AAPL        AMZN   
Date                                                                    
2012-06-01    0.273951  100.006142    1.876667  520987600  79030000.0   
2012-06-04    0.268234   99.959328    1.858667  556995600  85992000.0   
2012-06-05    0.276009  100.716232    1.860667  388214400  70878000.0   
2012-06-06    0.283326  102.979172    1.948000  401455600  54202000.0   
2012-06-07    0.271893  103.041573    1.928667  379766800  70078000.0   
...                ...         ...         ...        ...         ...   
2026-09-10  218.360001  757.830017  363.559998   70011900  25484800.0   
2026-09-11  218.289993  764.289978  365.440002   50716900  26724100.0   
2026-09-14  210.960007  760.880005  358.970001   39269100  34352800.0   
2026-09-15  212.169998  757.390015  356.579987   31748200  36275400.0   
2026-09-16  213.899994  754.049988  358.079987   35920400  33318500.0   

Price                                                                  \
Ticker            GOOGL        META      MSFT         NVDA        SPY   
Date                                                                    
2012-06-01  122193684.0  41855500.0  56634300  440984000.0  253240900   
2012-06-04   97210692.0  35230300.0  47926300  432856000.0  202545800   
2012-06-05   93502404.0  42473400.0  45715400  365224000.0  164149400   
2012-06-06   83748168.0  61489200.0  46860500  368968000.0  184202800   
2012-06-07   70269660.0  26159500.0  37792800  526780000.0  184772700   
...                 ...         ...       ...          ...        ...   
2026-09-10   23557600.0  21531900.0  16038800  105768000.0   42740400   
2026-09-11   24708300.0  16925000.0  14510500   89060100.0   45512700   
2026-09-14   35905400.0  19332300.0  23091900  132267200.0   43992400   
2026-09-15   21928600.0  19502800.0  17794600   88059700.0   46195000   
2026-09-16   18860000.0  17168100.0  16612700   96079300.0   58959700   

Price                   
Ticker            TSLA  
Date                    
2012-06-01  13287000.0  
2012-06-04  15463500.0  
2012-06-05   9463500.0  
2012-06-06  13648500.0  
2012-06-07   7381500.0  
...                ...  
2026-09-10  29667200.0  
2026-09-11  30153000.0  
2026-09-14  32477200.0  
2026-09-15  30317700.0  
2026-09-16  32031100.0  

[3593 rows x 16 columns]

In [5]:
df2 = df.drop(columns=["Volume"]).droplevel(0, axis=1)
df2


Ticker,AAPL,AMZN,GOOGL,META,MSFT,NVDA,SPY,TSLA
Date,,,,,,,,
2012-06-01,16.779005,10.411000,14.154103,27.478691,22.432699,0.273951,100.006142,1.876667
2012-06-04,16.877703,10.728500,14.342749,26.665831,22.511545,0.268234,99.959328,1.858667
2012-06-05,16.834030,10.660500,14.139973,25.644798,22.480003,0.276009,100.716232,1.860667
2012-06-06,17.092157,10.882000,14.391830,26.576614,23.142332,0.283326,102.979172,1.948000
2012-06-07,17.099934,10.940000,14.333823,26.080971,23.047722,0.271893,103.041573,1.928667
...,...,...,...,...,...,...,...,...
2026-09-10,326.570007,251.889999,332.600006,644.380005,492.440002,218.360001,757.830017,363.559998
2026-09-11,332.269989,256.779999,338.500000,648.030029,495.630005,218.289993,764.289978,365.440002
2026-09-14,333.079987,253.539993,349.390015,665.599976,505.410004,210.960007,760.880005,358.970001


In [6]:
normalized = ((df2 / df2.iloc[0])-1)*100
# Equal weight M7 average (ideal/theoretical distribution — 1/7 each)
normalized["M7"]= normalized[["GOOGL", "AMZN", "AAPL", "META", "MSFT","NVDA","TSLA"]].mean(axis=1)
# Simulating 1000 randomly weighted M7 portfolios and averaging to approximate realistic investor returns
mag7 = ["GOOGL", "AMZN", "AAPL", "META", "MSFT", "NVDA", "TSLA"]

n_simulations = 1
results = []

for _ in range(n_simulations):
    weights = np.random.dirichlet(np.ones(7))
    portfolio_return = (normalized[mag7] * weights).sum(axis=1)
    results.append(portfolio_return)

simulations = pd.DataFrame(results).T
simulations.index = normalized.index

# One average line
normalized["IRL-M7"] = simulations.mean(axis=1)
normalized.head()

Ticker,AAPL,AMZN,GOOGL,META,MSFT,NVDA,SPY,TSLA,M7,IRL-M7
Date,,,,,,,,,,
2012-06-01,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
2012-06-04,0.588221,3.049660,1.332796,-2.958149,0.351478,-2.086798,-0.046811,-0.959148,-0.097420,-2.121751
2012-06-05,0.327940,2.396497,-0.099834,-6.673873,0.210871,0.751251,0.710047,-0.852577,-0.562818,-4.767731
2012-06-06,1.866334,4.524058,1.679564,-3.282823,3.163386,3.422334,2.972848,3.801044,2.167699,-1.469300
2012-06-07,1.912679,5.081158,1.269737,-5.086561,2.741635,-0.751251,3.035244,2.770866,1.134038,-2.977345


In [8]:
top7_2012 = ["AAPL", "XOM", "MSFT", "IBM", "GE", "CVX", "BRK-B"]
data_2012 = yf.download(top7_2012, start="2012-06-01", end="2026-09-17")["Close"]


[*********************100%***********************]  7 of 7 completed


In [11]:
normalized_1= ((data_2012/data_2012.iloc[0])-1)*100
normalized_1["2012-M7"]= normalized_1[top7_2012].mean(axis= 1)
normalized_1.head()

Ticker,AAPL,BRK-B,CVX,GE,IBM,MSFT,XOM,2012-M7
Date,,,,,,,,
2012-06-01,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
2012-06-04,0.588221,0.025315,0.176270,-2.103571,-0.285589,0.351495,-0.115496,-0.194765
2012-06-05,0.328031,0.126558,0.082946,-1.618149,0.063489,0.210914,-0.410675,-0.173841
2012-06-06,1.866369,2.100739,3.516176,1.833865,2.596796,3.163455,2.900446,2.568264
2012-06-07,1.912703,2.075433,4.128158,2.481135,2.834796,2.741653,3.554960,2.818405


In [28]:
fig = go.Figure()

# SPY
fig.add_scatter(
    x=normalized.index,
    y=normalized["SPY"],
    name="SPY",
    mode="lines"
)

# Magnificent 7 from 2012
fig.add_scatter(
    x=normalized_1.index,
    y=normalized_1["2012-M7"],
    name="M7 (2012)",
    mode="lines"
)
# Current Magnificent 7:

fig.add_scatter(
    x=normalized.index,
    y=normalized["M7"],
    name= "M7",
    mode="lines")

fig.add_scatter(
    x=normalized.index,
    y=normalized["IRL-M7"],
    name="PF-M7",
    mode="lines"
)
fig.update_layout(
    title=dict(
        text="SPY vs Magnificent 7 (2012)",
        font=dict(size=24)
    ),
    xaxis_title="Date",
    yaxis_title="Normalized Performance (%)"
)



fig.show()

In [36]:
fig4 = px.line(normalized,x=normalized.index,y=["GOOGL", "AMZN", "AAPL", "META", "MSFT","NVDA","TSLA"])
fig4.update_traces(opacity=0.7)
fig4.update_layout(title=dict(text="The magnificent seven's growth over-time",font=dict(size=24)),
                   yaxis_title="Change(%)")
fig4.show()

In [23]:
# Resample to yearly
normalized_yearly = normalized.resample("YE").last()

# Reshape to long format
normalized_long = normalized_yearly.reset_index().melt(
    id_vars="Date",
    var_name="Company",
    value_name="Return"
)

normalized_long["Date"] = normalized_long["Date"].dt.strftime("%Y-%m")


# Events
events = {
    "2013-12": "2013: Steve Ballmer announces retirement",
    "2014-12": "2014: Satya Nadella becomes Microsoft CEO",
    "2015-12": "Microsoft under new leadership",
    "2016-12": "NVDA & TSLA take over the market",
    "2022-06": "2022: Inflation + Fed rate hikes + Ukraine war",
}


# Create animation
fig5 = px.bar(
    normalized_long,
    x="Company",
    y="Return",
    animation_frame="Date",
    range_y=[0.01, normalized_long["Return"].max() * 1.3],
    log_y=True,
    title="Company performance from 2012"
)


# Add / remove annotations for each frame
for frame in fig5.frames:

    year = frame.name[:4]

    # Show event from 2013 through 2022
    if year in ["2013", "2014", "2015", "2016","2022"]:

        frame.layout = {
            "annotations": [
                {
                    "x": 0.5,
                    "y": 0.95,
                    "xref": "paper",
                    "yref": "paper",
                    "text": events.get(
                        frame.name,
                        "2022-06: 2022: Inflation + Fed rate hikes + Ukraine war"
                    ),
                    "showarrow": False,
                    "font": {
                        "size": 18
                    }
                }
            ]
        }

    # Remove annotation after 2016
    else:
        frame.layout = {
            "annotations": []
        }


# Slow down animation
fig5.layout.updatemenus[0].buttons[0].args[1]["frame"]["duration"] = 1500
fig5.layout.updatemenus[0].buttons[0].args[1]["transition"]["duration"] = 800

fig5.show()

In [30]:
print("Current M7:", normalized["M7"].iloc[-1].round(2), "%")
print("2012 Top 7:", normalized_1["2012-M7"].iloc[-1].round(2), "%")
print("SPY:", normalized["SPY"].iloc[-1].round(2), "%")

Current M7: 15408.9 %
2012 Top 7: 794.56 %
SPY: 654.0 %


In [31]:
individual_growth={}
for ticker in mag7:
    data=yf.download(ticker,period="max")["Close"].squeeze().dropna()
    normalized_ticker= ((data/data.iloc[0])-1 )*100
    individual_growth[ticker]= normalized_ticker
df_individual=pd.DataFrame(individual_growth)

[*********************100%***********************]  1 of 1 completed
[*********************100%***********************]  1 of 1 completed
[*********************100%***********************]  1 of 1 completed
[*********************100%***********************]  1 of 1 completed
[*********************100%***********************]  1 of 1 completed
[*********************100%***********************]  1 of 1 completed
[*********************100%***********************]  1 of 1 completed


In [32]:
final_returns = df_individual.iloc[-1]

# Calculate years public
years = df_individual.apply(lambda x: x.dropna().shape[0] / 252).round(1)

fig7 = px.bar(x=final_returns.index, y=final_returns.values,
              title="Total Return Since IPO",
              labels={"x": "Company", "y": "% Return"},
              log_y=True,
              text=[f"{y} yrs" for y in years])

fig7.update_traces(textposition="outside")
fig7.show()

In [34]:
growth_velocity = ((1 + final_returns/100) ** (1/years) - 1) * 100
fig_velocity=px.bar(growth_velocity,x=growth_velocity.index,y=growth_velocity.values,labels={"x": "Company", "y": "Annualized Return (CAGR %)"})

fig_velocity.update_layout(title=dict(text="Annualized Return (CAGR) Since IPO", font=dict(size=24)))

fig_velocity.show()